# AO3 Fandom Spread per Tag

For every value of a tag field (default `additional_tags`), how many **distinct fandoms** it appears on, plus its top co-occurring fandoms. A cross-cutting trope like `Angst` spans many fandoms; a niche tag only a few. Rows are sorted most-cross-cutting first.

Reads a local `ao3_tag_metadata.csv` (from `ao3_tag_scraper.py`); no network access.

In [ ]:
# Installs any of this notebook's dependencies that aren't already present.
# Safe to re-run.
%pip install -q pandas

In [ ]:
import pandas as pd

## Configuration

In [ ]:
INPUT = "ao3_tag_metadata.csv"
FIELD = "additional_tags"   # tag field to analyze (any tag-bearing field works)
TOP_N = 3                   # top co-occurring fandoms to list per tag
OUT = "ao3_tag_fandom_spread.csv"

DELIMITER = ", "
ALL_METADATA_FIELDS = ["rating", "warnings", "category", "fandom",
                        "relationship", "character", "additional_tags"]

## Load metadata + shared helpers

In [ ]:
# copied from ao3_tag_visualizer.py -- keep in sync if that file changes
def load_metadata(input_csv):
    df = pd.read_csv(input_csv, dtype=str, keep_default_na=False)
    # Drop exact-duplicate (tag, work_id) rows (re-scrape artifacts); a work
    # found under several DIFFERENT seed tags is kept.
    df = df.drop_duplicates(subset=["tag", "work_id"], keep="first").reset_index(drop=True)
    return df


def split_values(cell, delimiter=DELIMITER):
    if not cell:
        return []
    values = [v.strip() for v in cell.split(delimiter) if v.strip()]
    return list(dict.fromkeys(values))


def explode_field(df, field):
    exploded = df[["tag", "work_id", field]].copy()
    exploded[field] = exploded[field].map(split_values)
    exploded = exploded.explode(field)
    exploded = exploded[exploded[field].notna() & (exploded[field] != "")]
    return exploded


def build_document_tag_table(df, fields=ALL_METADATA_FIELDS):
    deduped = df.drop_duplicates(subset="work_id", keep="first")
    tables = []
    for field in fields:
        exploded = explode_field(deduped, field)
        exploded = exploded.rename(columns={field: "value"})
        exploded["tag_id"] = field + "::" + exploded["value"]
        tables.append(exploded[["work_id", "tag_id"]])
    return pd.concat(tables, ignore_index=True)


df = load_metadata(INPUT)
df.head()

## Fandom spread

In [ ]:
# compute_fandom_summary copied from ao3_tag_fandom_labels.py; fandom_spread from
# ao3_tag_fandom_spread.py -- keep in sync if those files change
def compute_fandom_summary(df, tag_ids, top_n):
    tag_table = build_document_tag_table(df, fields=ALL_METADATA_FIELDS)
    tag_table = tag_table[tag_table["tag_id"].isin(tag_ids)]
    tag_totals = tag_table.groupby("tag_id").size()

    # Dedup by work_id before exploding fandom -- the scraper emits one row per
    # (seed tag, work), so without this a work found via k seed tags would
    # contribute its fandom k times while tag_totals counts it once.
    deduped = df.drop_duplicates(subset="work_id", keep="first")
    fandom_table = explode_field(deduped, "fandom")[["work_id", "fandom"]]

    merged = tag_table.merge(fandom_table, on="work_id")
    counts = merged.groupby(["tag_id", "fandom"]).size().reset_index(name="count")
    counts["pct"] = counts["count"] / counts["tag_id"].map(tag_totals) * 100

    n_fandoms = counts.groupby("tag_id")["fandom"].nunique()

    counts = counts.sort_values(["tag_id", "count", "fandom"], ascending=[True, False, True])
    top = counts.groupby("tag_id", sort=False).head(top_n)
    top = top.assign(entry=top["fandom"] + " (" + top["pct"].round(0).astype(int).astype(str) + "%)")
    labels = top.groupby("tag_id", sort=False)["entry"].apply(", ".join)

    ordered = list(tag_ids)
    return pd.DataFrame({
        "tag_id": ordered,
        "n_fandoms": [int(n_fandoms.get(t, 0)) for t in ordered],
        "top_fandoms": [labels.get(t, "") for t in ordered],
    })


OUT_COLUMNS = ["field", "value", "n_works", "n_fandoms", "top_fandoms"]


def fandom_spread(df, field="additional_tags", top_n=3):
    tag_table = build_document_tag_table(df, fields=[field])
    if tag_table.empty:
        return pd.DataFrame(columns=OUT_COLUMNS)
    n_works = tag_table.groupby("tag_id").size()
    tag_ids = set(tag_table["tag_id"].unique())
    summary = compute_fandom_summary(df, tag_ids, top_n)
    summary = summary.assign(
        field=summary["tag_id"].str.split("::", n=1).str[0],
        value=summary["tag_id"].str.split("::", n=1).str[1],
        n_works=summary["tag_id"].map(n_works).astype(int),
    )
    summary = summary.sort_values(
        ["n_fandoms", "n_works", "value"], ascending=[False, False, True])
    return summary[OUT_COLUMNS].reset_index(drop=True)


spread = fandom_spread(df, field=FIELD, top_n=TOP_N)
spread.to_csv(OUT, index=False)
print(f"wrote {OUT} ({len(spread)} {FIELD} values)")
spread.head(20)

## Done